In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType,DateType

schema=StructType([
    StructField("product_id",IntegerType(),True),
    StructField("customer_id",StringType(),True),
    StructField("order_date",DateType(),True),
    StructField("location",StringType(),True),
    StructField("source_order",StringType(),True)
])

sales_df=spark.read.format("csv").option("interschema","true").schema(schema).load("/Volumes/workspace/default/sales_analysis/sales.csv.txt")
display(sales_df)

product_id,customer_id,order_date,location,source_order
1,A,2023-01-01,India,Swiggy
2,A,2022-01-01,India,Swiggy
2,A,2023-01-07,India,Swiggy
3,A,2023-01-10,India,Restaurant
3,A,2022-01-11,India,Swiggy
3,A,2023-01-11,India,Restaurant
2,B,2022-02-01,India,Swiggy
2,B,2023-01-02,India,Swiggy
1,B,2023-01-04,India,Restaurant
1,B,2023-02-11,India,Swiggy


In [0]:
from pyspark.sql.functions import month,year,quarter
sales_df=sales_df.withColumn("month",month(sales_df.order_date)).withColumn("year",year(sales_df.order_date)).withColumn("quarter",quarter(sales_df.order_date))
display(sales_df)

product_id,customer_id,order_date,location,source_order,month,year,quarter
1,A,2023-01-01,India,Swiggy,1,2023,1
2,A,2022-01-01,India,Swiggy,1,2022,1
2,A,2023-01-07,India,Swiggy,1,2023,1
3,A,2023-01-10,India,Restaurant,1,2023,1
3,A,2022-01-11,India,Swiggy,1,2022,1
3,A,2023-01-11,India,Restaurant,1,2023,1
2,B,2022-02-01,India,Swiggy,2,2022,1
2,B,2023-01-02,India,Swiggy,1,2023,1
1,B,2023-01-04,India,Restaurant,1,2023,1
1,B,2023-02-11,India,Swiggy,2,2023,1


In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType,DataType

schema=StructType([
    StructField("product_id",IntegerType(),True),
    StructField("product_name",StringType(),True),
    StructField("price",StringType(),True)
])

menu_df=spark.read.format("csv").option("interschema","true").schema(schema).load("/Volumes/workspace/default/sales_analysis/menu.csv.txt")
display(menu_df)

product_id,product_name,price
1,PIZZA,100
2,Chowmin,150
3,sandwich,120
4,Dosa,110
5,Biryani,80
6,Pasta,180


In [0]:
total_amount_spent=(sales_df.join(menu_df,'product_id').groupby('customer_id').agg({'price':'sum'}).orderBy('customer_id'))

display(total_amount_spent)

customer_id,sum(price)
A,4260.0
B,4440.0
C,2400.0
D,1200.0
E,2040.0


Databricks visualization. Run in Databricks to view.

In [0]:
total_amount_spent4food=(sales_df.join(menu_df,'product_id').groupby('product_name').agg({'price':'sum'}).orderBy('product_name'))

display(total_amount_spent4food)

product_name,sum(price)
Biryani,480.0
Chowmin,3600.0
Dosa,1320.0
PIZZA,2100.0
Pasta,1080.0
sandwich,5760.0


Databricks visualization. Run in Databricks to view.

In [0]:
df1=(sales_df.join(menu_df,'product_id').groupBy('month').agg({'price':'sum'}))

display(df1)

month,sum(price)
5,2960.0
1,2960.0
3,910.0
2,2730.0
6,2960.0
7,910.0
11,910.0


Databricks visualization. Run in Databricks to view.

In [0]:
df2=(sales_df.join(menu_df,'product_id').groupBy('year').agg({'price':'sum'}))

display(df2)

year,sum(price)
2022,4350.0
2023,9990.0


Databricks visualization. Run in Databricks to view.

In [0]:
df3=(sales_df.join(menu_df,'product_id').groupBy('quarter').agg({'price':'sum'}))

display(df3)

quarter,sum(price)
1,6600.0
3,910.0
2,5920.0
4,910.0


Databricks visualization. Run in Databricks to view.

In [0]:
from pyspark.sql.functions import count

df5=(sales_df.join(menu_df,'product_id').groupBy('product_id','product_name').agg(count('product_id')).alias('product_count'))

display(df5)

product_id,product_name,count(product_id)
6,Pasta,6
1,PIZZA,21
3,sandwich,48
2,Chowmin,24
5,Biryani,6
4,Dosa,12


Databricks visualization. Run in Databricks to view.

In [0]:
from pyspark.sql.functions import count

df6 = (
    sales_df
    .join(menu_df, "product_id")
    .groupBy("product_id", "product_name")
    .agg(count("product_id").alias("product_count"))
    .orderBy("product_count", ascending=False)
    .drop("product_id")
    .limit(5)
)

display(df6)

product_name,product_count
sandwich,48
Chowmin,24
PIZZA,21
Dosa,12
Pasta,6


Databricks visualization. Run in Databricks to view.

In [0]:
from pyspark.sql.functions import count

df7 = (
    sales_df
    .join(menu_df, "product_id")
    .groupBy("product_id", "product_name")
    .agg(count("product_id").alias("product_count"))
    .orderBy("product_count", ascending=False)
    .drop("product_id")
    .limit(1)
)

display(df7)

product_name,product_count
sandwich,48


Databricks visualization. Run in Databricks to view.

In [0]:
from pyspark.sql.functions import countDistinct

df9 = (sales_df.filter(sales_df.source_order == 'Restaurant').groupBy('customer_id').agg(countDistinct('order_date')))

display(df9)


customer_id,count(DISTINCT order_date)
B,6
A,6
E,5
D,1
C,3


Databricks visualization. Run in Databricks to view.

In [0]:
df10=(sales_df.join(menu_df,"product_id").groupBy('location').agg({'price':'sum'}))
display(df10)

location,sum(price)
USA,2460.0
India,4860.0
UK,7020.0


Databricks visualization. Run in Databricks to view.

In [0]:
df11=(sales_df.join(menu_df,"product_id").groupBy('source_order').agg({'price':'sum'}))
display(df11)

source_order,sum(price)
zomato,4920.0
Restaurant,3090.0
Swiggy,6330.0


Databricks visualization. Run in Databricks to view.